# **Deep Learning - PatchTST from Scratch**

---

* This repository contains the implementation of the PatchTST (Patch Time Series Transformer) architecture from scratch. It focuses on multivariate time series forecasting by applying patching techniques and transformer-based models to efficiently capture long-term dependencies.

**Authors:**

Lopez Medina, Sebastian  
[sebastian.lopez@utec.edu.pe](mailto:sebastian.lopez@utec.edu.pe)

Nieto Espinoza, Brajan  
[brajan.nieto@utec.edu.pe](mailto:brajan.nieto@utec.edu.pe)

Tapia Chasquibol, Mateo  
[mateo.tapia@utec.edu.pe](mailto:mateo.tapia@utec.edu.pe)

---

# 0. Configuración de Entorno, Librerías y Dataser

* **Librerías:** Importación de PyTorch, herramientas de manejo de datos y visualización.
* **Reproducibilidad:** Configuración de `SEED` y detección de aceleración por hardware (CUDA).
* **Fuentes de Datos:** Definición de diccionarios de URLs y constantes para la selección de variables del dataset.

In [ ]:
# ============================================================================
# 0. LIBRERÍAS
# ============================================================================
import math
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================================
# 0.1 REPRODUCIBILIDAD Y DISPOSITIVO
# ============================================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Dispositivo: {DEVICE}")

[INFO] Dispositivo: cuda


In [ ]:
# ============================================================================
# 0.2 URLS DE DATOS Y CONSTANTES
# ============================================================================
DATA_URLS = {
    "ETTh1": "https://raw.githubusercontent.com/BrajanNieto/DeepLearning/main/ETTh1.csv",
    "ETTh2": "https://raw.githubusercontent.com/BrajanNieto/DeepLearning/main/ETTh2.csv",
    "ETTm1": "https://raw.githubusercontent.com/BrajanNieto/DeepLearning/main/ETTm1.csv",
    "ETTm2": "https://raw.githubusercontent.com/BrajanNieto/DeepLearning/main/ETTm2.csv",
}

SOURCE_DATASET = "ETTh1"
TARGET_DATASETS = ["ETTh2", "ETTm1", "ETTm2"]

FEATURE_COLS = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]
TARGET_COL = "OT"

# 1. Clase Dataset: Sliding Window

Definición de la clase `ETTDataset` encargada de la preparación de los datos mediante la técnica de (*sliding window*). Esta estructura es fundamental para modelos de series temporales, permitiendo transformar datos tabulares en tensores de PyTorch con secuencias históricas y horizontes de predicción.

* **Entrada (x):** Ventana de contexto histórico de longitud `seq_len`.
* **Salida (y):** Horizonte futuro o *ground truth* de longitud `pred_len`.
* **Procesamiento:** Conversión automática a tensores de punto flotante de 32 bits para compatibilidad con la GPU.

In [ ]:
# ============================================================================
# 1. CLASE DATASET — Ventanas deslizantes
# ============================================================================
class ETTDataset(Dataset):
    """
    Genera pares (x, y) mediante ventana deslizante.
        x : (seq_len, n_channels)   — ventana de contexto histórico
        y : (pred_len, n_channels)  — horizonte futuro (ground truth)
    """

    def __init__(self, data: np.ndarray, seq_len: int, pred_len: int):
        self.data = data.astype(np.float32)
        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, index):
        x_start = index
        x_end = x_start + self.seq_len
        y_end = x_end + self.pred_len
        x = torch.from_numpy(self.data[x_start:x_end])   # (seq_len, C)
        y = torch.from_numpy(self.data[x_end:y_end])      # (pred_len, C)
        return x, y

# 2. Creación de DataLoaders

Implementación de la función `make_DataLoader`, la cual gestiona la partición de los datos en conjuntos de entrenamiento, validación y prueba siguiendo un **orden cronológico estricto**. Esta división es vital en series temporales para evitar el *data leakage* (fuga de información del futuro).

* **Particiones:** Se utiliza una distribución de **80% entrenamiento**, **10% validación** y **10% prueba**.
* **Barajado (Shuffle):** Solo se activa para el conjunto de entrenamiento, mejorando la generalización del modelo sin romper la estructura secuencial de cada ventana individual.
* **Carga por Lotes:** Configuración de `DataLoader` para optimizar el paso de datos a la memoria de la GPU durante el entrenamiento.

In [ ]:
# ============================================================================
# 2. CREACIÓN DE DATALOADERS  (Train 80% | Valid 10% | Test 10%)
# ============================================================================
def make_DataLoader(df: pd.DataFrame, seq_len: int, pred_len: int,
                    batch_size: int, columns: list):
    """
    Divide el DataFrame en orden cronológico estricto.
    Returns: (train_loader, valid_loader, test_loader)
    """
    data = df[columns].values
    n = len(data)

    n_train = int(n * 0.8)
    n_valid = int(n * 0.1)

    train_data = data[:n_train]
    valid_data = data[n_train : n_train + n_valid]
    test_data  = data[n_train + n_valid :]

    train_ds = ETTDataset(train_data, seq_len, pred_len)
    valid_ds = ETTDataset(valid_data, seq_len, pred_len)
    test_ds  = ETTDataset(test_data,  seq_len, pred_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, drop_last=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size,
                              shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, drop_last=False)
    return train_loader, valid_loader, test_loader

# 3. Arquitectura del Modelo PatchTST



Implementación completa de la arquitectura PatchTST utilizando PyTorch, construida desde sus bloques más fundamentales siguiendo las especificaciones del paper original. Esta sección se divide en tres componentes clave:

* **Atención Multi-Cabezal (Multi-Head Attention):** Implementa el núcleo del Transformer, calculando la atención de producto punto escalado mediante la fórmula matemática $O = \text{Softmax}(\frac{Q K^T}{\sqrt{d_k}}) V$.
* **Capa Encoder (PatchTSTEncoderLayer):** Un bloque de Transformer modificado que combina la capa de atención con redes *Feed-Forward* (utilizando activación GELU) y conexiones residuales con *Batch Normalization*.
* **Modelo Principal (PatchTST):** Orquesta el flujo de datos aplicando la estrategia de **Independencia de Canales**. El proceso secuencial que sigue cada serie de tiempo es:
  1. **Normalización de Instancia:** Centra y escala los datos (guardando $\mu$ y $\sigma$).
  2. **Patching:** Divide la serie en ventanas más pequeñas (parches) mediante la operación `unfold`.
  3. **Proyección y Posición:** Proyección lineal de los parches y suma del *Positional Encoding* aprendible.
  4. **Procesamiento:** Paso de la secuencia de parches por las capas del *Transformer Encoder*.
  5. **Predicción:** Aplanado (*Flatten*) y paso por el *Linear Head* para proyectar el horizonte temporal futuro.
  6. **Desnormalización:** Restaura la escala original de las predicciones usando los $\mu$ y $\sigma$ guardados.

In [ ]:
# ============================================================================
# 3. MODELO PatchTST — from scratch
# ============================================================================

# ---------- 3A. Multi-Head Attention ----
class MultiHeadAttention(nn.Module):
    """
    Sección 3.5-A:  Q, K, V con matrices aprendibles.
    O = Softmax( Q·K^T / √d_k ) · V
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """x : (B, N, D) → out : (B, N, D)"""
        B, N, D = x.shape

        Q = self.W_Q(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, N, D)
        out = self.W_O(out)
        return out


# ---------- 3B. Encoder Layer (una capa) -----------------------------------
class PatchTSTEncoderLayer(nn.Module):
    """
    Sección 3.5 completa:
        A. Multi-Head Attention
        B. Add & Norm 1 (BatchNorm)
        C. FFN (Linear → GELU → Linear)
        D. Add & Norm 2 (BatchNorm)
    """

    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 dropout: float = 0.2):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.BatchNorm1d(d_model)

        self.ff_linear1 = nn.Linear(d_model, d_ff)
        self.ff_linear2 = nn.Linear(d_ff, d_model)
        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.BatchNorm1d(d_model)

    def forward(self, x):
        """x : (B, N, D) → z : (B, N, D)"""
        # A. Atención
        O = self.dropout1(self.attn(x))
        # B. Add & Norm 1
        residual = (x + O).transpose(1, 2)          # (B, D, N)
        B_out = self.norm1(residual).transpose(1, 2) # (B, N, D)
        # C. Feed Forward
        F_out = self.ff_linear2(F.gelu(self.ff_linear1(B_out)))
        F_out = self.dropout2(F_out)
        # D. Add & Norm 2
        residual2 = (B_out + F_out).transpose(1, 2)
        z = self.norm2(residual2).transpose(1, 2)
        return z


# ---------- 3C. PatchTST completo (canal-independiente) --------------------
class PatchTST(nn.Module):
    """
    Modelo canal-independiente.  Flujo por canal i:
        3.1  Instance Normalization     →  guardar μ, σ
        3.2  Patching                   →  unfold
        3.3  Linear Projection          →  W_p (P → D)
        3.4  Positional Encoding        →  W_pos (learnable)
        3.5  Transformer Encoder × n
        3.6  Flatten                    →  (D·N)
        3.7  Linear Head                →  (D·N) → pred_len
        3.8  Denormalization            →  revertir con μ, σ
    """

    def __init__(self, n_channels: int, seq_len: int, pred_len: int,
                 patch_len: int = 16, stride: int = 8,
                 d_model: int = 128, n_heads: int = 4, e_layers: int = 3,
                 d_ff: int = 256, dropout: float = 0.2):
        super().__init__()
        self.n_channels = n_channels
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        self.d_model = d_model
        # 3.2 Número de parches:  N = floor((L+pad - P) / S) + 1
        self.pad_len = stride
        L_padded = seq_len + self.pad_len
        self.n_patches = (L_padded - patch_len) // stride + 1
        # 3.3 Proyección lineal:  W_p ∈ R^{D × P}
        self.patch_proj = nn.Linear(patch_len, d_model, bias=False)
        # 3.4 Positional Encoding:  W_pos ∈ R^{1 × N × D}
        self.pos_encoding = nn.Parameter(
            torch.randn(1, self.n_patches, d_model) * 0.02
        )
        # 3.5 Encoder
        self.encoder_layers = nn.ModuleList([
            PatchTSTEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(e_layers)
        ])
        # 3.6 + 3.7 Flatten → Head
        self.flatten_dim = d_model * self.n_patches
        self.head = nn.Linear(self.flatten_dim, pred_len, bias=True)

        self.dropout_embed = nn.Dropout(dropout)

    def forward(self, x):
        """
        x : (B, seq_len, C) → pred : (B, pred_len, C)
        """
        B, L, C = x.shape

        # 3.1 Instance Normalization
        mu = x.mean(dim=1, keepdim=True)
        sigma = x.std(dim=1, keepdim=True) + 1e-8
        x_norm = (x - mu) / sigma
        # Canal-independiente: (B, L, C) → (B*C, L)
        x_ci = x_norm.permute(0, 2, 1).reshape(B * C, L)

        # 3.2 Patching
        x_padded = F.pad(x_ci, (0, self.pad_len), mode='replicate')
        patches = x_padded.unfold(dimension=1, size=self.patch_len,
                                  step=self.stride)       # (B*C, N, P)
        # 3.3 Linear Projection
        x_d = self.patch_proj(patches)                     # (B*C, N, D)
        # 3.4 Positional Encoding
        x_d = self.dropout_embed(x_d + self.pos_encoding)
        # 3.5 Transformer Encoder
        z = x_d
        for layer in self.encoder_layers:
            z = layer(z)
        # 3.6 Flatten
        v = z.reshape(B * C, -1)                           # (B*C, D·N)
        # 3.7 Linear Head
        x_hat = self.head(v)                               # (B*C, pred_len)
        x_hat = x_hat.view(B, C, self.pred_len).permute(0, 2, 1)
        # 3.8 Denormalization
        x_hat_real = x_hat * sigma + mu
        return x_hat_real

# 4. Funciones Core de Entrenamiento y Evaluación



Esta sección define el motor principal para el aprendizaje del modelo, encapsulando la lógica de inicialización, optimización y validación en PyTorch.

* **`build_model`**: Instancia la arquitectura PatchTST y la envía a la memoria del acelerador de hardware (GPU/CPU).
* **`train_epoch`**: Ejecuta la propagación hacia adelante y hacia atrás (*backpropagation*) para un lote de datos. Incluye la técnica de *Gradient Clipping* (`clip_grad_norm_`) para asegurar la estabilidad del entrenamiento y evitar explosiones en los gradientes.
* **`evaluate`**: Función de inferencia (sin cálculo de gradientes). Aísla específicamente la variable objetivo (*target*) para calcular las métricas de rendimiento absolutas: el Error Cuadrático Medio (**MSE**) y el Error Absoluto Medio (**MAE**).
* **`train_full`**: Orquesta el bucle iterativo a través de las épocas. Integra un *Learning Rate Scheduler* y un mecanismo de **Early Stopping** (parada temprana) que monitorea la pérdida de validación para guardar únicamente los mejores pesos del modelo, previniendo así el sobreajuste (*overfitting*).

In [ ]:
# ============================================================================
# 4. FUNCIONES DE ENTRENAMIENTO Y EVALUACIÓN
# ============================================================================

def build_model(n_channels, seq_len, pred_len, patch_len, stride,
                d_model, n_heads, e_layers, d_ff, dropout):
    """Instancia el modelo y lo envía al dispositivo."""
    return PatchTST(
        n_channels=n_channels, seq_len=seq_len, pred_len=pred_len,
        patch_len=patch_len, stride=stride, d_model=d_model,
        n_heads=n_heads, e_layers=e_layers, d_ff=d_ff, dropout=dropout
    ).to(DEVICE)


def train_epoch(model, loader, optimizer, criterion):
    """Entrena una época. Retorna pérdida promedio."""
    model.train()
    total_loss, n = 0.0, 0
    for x_b, y_b in loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x_b)
        loss = criterion(pred, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, target_idx: int = None):
    """
    Evalúa en un loader. Métricas MSE/MAE solo sobre canal target_idx.
    Returns: (mse, mae, avg_loss, preds_tensor, trues_tensor)
    """
    model.eval()
    all_preds, all_trues = [], []
    total_loss, n = 0.0, 0

    for x_b, y_b in loader:
        x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
        pred = model(x_b)
        total_loss += criterion(pred, y_b).item()
        n += 1
        all_preds.append(pred.cpu())
        all_trues.append(y_b.cpu())

    preds = torch.cat(all_preds, dim=0)
    trues = torch.cat(all_trues, dim=0)

    if target_idx is not None:
        p, t = preds[:, :, target_idx], trues[:, :, target_idx]
    else:
        p = preds.squeeze(-1) if preds.shape[-1] == 1 else preds[:, :, -1]
        t = trues.squeeze(-1) if trues.shape[-1] == 1 else trues[:, :, -1]

    mse = F.mse_loss(p, t).item()
    mae = F.l1_loss(p, t).item()
    return mse, mae, total_loss / max(n, 1), preds, trues


def train_full(model, train_ld, valid_ld, optimizer, criterion,
               scheduler, epochs, patience, target_idx, label=""):
    """
    Loop completo de entrenamiento con early stopping.
    Returns: (best_state_dict, train_losses, valid_losses)
    """
    train_losses, valid_losses = [], []
    best_val, best_state, pat_count = float("inf"), None, 0

    for epoch in range(1, epochs + 1):
        t_loss = train_epoch(model, train_ld, optimizer, criterion)
        _, _, v_loss, _, _ = evaluate(model, valid_ld, criterion, target_idx)
        scheduler.step()

        train_losses.append(t_loss)
        valid_losses.append(v_loss)

        if v_loss < best_val:
            best_val = v_loss
            best_state = copy.deepcopy(model.state_dict())
            pat_count = 0
        else:
            pat_count += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f"      [{label}] Epoch {epoch:3d}/{epochs}  "
                  f"Train: {t_loss:.6f}  Valid: {v_loss:.6f}")

        if pat_count >= patience:
            print(f"      [{label}] Early stopping en epoch {epoch}")
            break

    return best_state, train_losses, valid_losses

# 5. Fase 1: Pre-entrenamiento en el Dataset Origen (ETTh1)



Definición de la función `pretrain_on_source`, encargada de generar los **pesos fundacionales** del modelo. Esta etapa es el primer pilar de la estrategia de *Transfer Learning*, donde PatchTST aprende los patrones y representaciones subyacentes de las series de tiempo usando el dataset histórico de origen (`ETTh1`).

* **Iteración Exhaustiva:** Ejecuta ciclos de entrenamiento independientes para cada horizonte de predicción temporal (`pred_len`) y para ambos modos operativos (Univariado y Multivariado).
* **Estrategia de Optimización:** Implementa el optimizador **Adam** combinado con un planificador de tasa de aprendizaje **Cosine Annealing** (`CosineAnnealingLR`) para asegurar una convergencia suave y evitar estancamientos en mínimos locales.
* **Almacenamiento de Pesos:** Guarda en memoria (`pretrained_weights`) el estado óptimo de la red (`state_dict`) alcanzado antes del *Early Stopping*. Estos pesos pre-entrenados serán el punto de partida exacto para la siguiente fase.
* **Gestión de Memoria:** Incorpora limpieza activa de la VRAM (`torch.cuda.empty_cache()`) al finalizar cada ciclo, paso indispensable para evitar el error *CUDA Out of Memory* al entrenar múltiples instancias secuencialmente.

In [ ]:
# ============================================================================
# 5. FASE 1 — PRE-ENTRENAMIENTO EN DATASET (ETTh1)
# ============================================================================

def pretrain_on_source(df_source: pd.DataFrame, pred_lens: list,
                       config: dict):
    """
    Pre-entrena un modelo PatchTST en ETTh1 para cada combinación de
    (modo, pred_len).  Guarda los state_dicts como "pesos fundacionales".

    Returns:
        pretrained_weights : dict
            Clave: "{modo}_pl{pred_len}"
            Valor: state_dict del mejor modelo
        pretrain_histories : dict con curvas de loss del pre-entrenamiento
        pretrain_metrics   : list de dicts con MSE/MAE en test de ETTh1
    """
    pretrained_weights = {}
    pretrain_histories = {}
    pretrain_metrics = []

    for pred_len in pred_lens:
        print(f"\n  {'─'*60}")
        print(f"  Pre-entrenando para pred_len = {pred_len}")
        print(f"  {'─'*60}")

        for mode in ["Multivariado", "Univariado"]:
            if mode == "Multivariado":
                columns = FEATURE_COLS
                n_channels = len(columns)
                target_idx = columns.index(TARGET_COL)
            else:
                columns = [TARGET_COL]
                n_channels = 1
                target_idx = 0

            print(f"\n    Modo: {mode} ({n_channels} canal{'es' if n_channels > 1 else ''})")

            train_ld, valid_ld, test_ld = make_DataLoader(
                df_source, config["seq_len"], pred_len,
                config["batch_size"], columns
            )

            model = build_model(
                n_channels, config["seq_len"], pred_len,
                config["patch_len"], config["stride"],
                config["d_model"], config["n_heads"],
                config["e_layers"], config["d_ff"], config["dropout"]
            )
            n_params = sum(p.numel() for p in model.parameters())
            print(f"    Parámetros: {n_params:,}")

            optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
            criterion = nn.MSELoss()
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=config["epochs"]
            )

            best_state, t_hist, v_hist = train_full(
                model, train_ld, valid_ld, optimizer, criterion, scheduler,
                config["epochs"], config["patience"], target_idx,
                label=f"Pretrain-{mode[:5]}"
            )

            model.load_state_dict(best_state)
            mse, mae, _, _, _ = evaluate(model, test_ld, criterion, target_idx)
            print(f"    TEST ETTh1 →  MSE: {mse:.6f}   MAE: {mae:.6f}")

            key = f"{mode}_pl{pred_len}"
            pretrained_weights[key] = best_state
            pretrain_histories[key] = {"train": t_hist, "valid": v_hist}
            pretrain_metrics.append({
                "Dataset": SOURCE_DATASET, "pred_len": pred_len,
                "Modo": mode, "Estrategia": "Pretrain",
                "MSE (OT)": round(mse, 6), "MAE (OT)": round(mae, 6)
            })

            del model, optimizer, scheduler
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return pretrained_weights, pretrain_histories, pretrain_metrics

# 6. Fase 2: Transfer Learning (Scratch vs. Fine-Tuning)



Definición de `evaluate_on_target`, el núcleo experimental del proyecto. Esta función evalúa la eficacia de transferir el conocimiento aprendido en el dataset de origen (`ETTh1`) hacia nuevos dominios (datasets destino), comparando sistemáticamente dos enfoques:

* **Entrenamiento desde Cero (Scratch):** El modelo se inicializa con pesos aleatorios y entrena utilizando la tasa de aprendizaje estándar (`lr`), actuando como línea base de rendimiento.
* **Afinamiento Completo (Fine-Tuning):** El modelo se inicializa cargando los pesos pre-entrenados ("pesos fundacionales"). Se entrena ajustando todos los parámetros de la red, pero utilizando una tasa de aprendizaje reducida (`lr_ft`) para preservar las representaciones globales ya aprendidas.

El ciclo itera sobre todos los horizontes de predicción (`pred_len`) y modos (univariado/multivariado), almacenando métricas (MSE, MAE), el historial de pérdidas y las predicciones finales para su posterior visualización comparativa.

In [ ]:
# ============================================================================
# 6. FASE 2 — SCRATCH vs FINE-TUNING EN DATASETS DESTINO
# ============================================================================

def evaluate_on_target(df_target: pd.DataFrame, dataset_name: str,
                       pred_lens: list, config: dict,
                       pretrained_weights: dict):
    """
    Para un dataset destino, compara Scratch vs Fine-Tuning para cada
    (modo, pred_len).

    Scratch     : Inicialización aleatoria, lr = config['lr'].
    Fine-Tuning : Carga pesos pre-entrenados de ETTh1, lr = config['lr_ft'].
                  Entrena TODOS los parámetros (fine-tuning total).

    Returns:
        results, histories, test_preds
    """
    results = []
    histories = {}
    test_preds = {}

    for pred_len in pred_lens:
        print(f"\n  {'='*60}")
        print(f"  {dataset_name} | pred_len = {pred_len}")
        print(f"  {'='*60}")

        for mode in ["Multivariado", "Univariado"]:
            if mode == "Multivariado":
                columns = FEATURE_COLS
                n_channels = len(columns)
                target_idx = columns.index(TARGET_COL)
            else:
                columns = [TARGET_COL]
                n_channels = 1
                target_idx = 0

            train_ld, valid_ld, test_ld = make_DataLoader(
                df_target, config["seq_len"], pred_len,
                config["batch_size"], columns
            )

            weight_key = f"{mode}_pl{pred_len}"

            for strategy in ["Scratch", "Fine-Tuning"]:
                print(f"\n    {mode} | {strategy}")

                model = build_model(
                    n_channels, config["seq_len"], pred_len,
                    config["patch_len"], config["stride"],
                    config["d_model"], config["n_heads"],
                    config["e_layers"], config["d_ff"], config["dropout"]
                )

                # ---- Decidir inicialización y lr ----
                if strategy == "Fine-Tuning":
                    # Cargar pesos fundacionales pre-entrenados en ETTh1
                    model.load_state_dict(pretrained_weights[weight_key])
                    lr = config["lr_ft"]
                    max_epochs = config["epochs_ft"]
                    print(f"    [INFO] Pesos de ETTh1 cargados → "
                          f"Fine-Tuning total con lr={lr}")
                else:
                    lr = config["lr"]
                    max_epochs = config["epochs"]
                    print(f"    [INFO] Inicialización aleatoria → lr={lr}")

                optimizer = torch.optim.Adam(model.parameters(), lr=lr)
                criterion = nn.MSELoss()
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=max_epochs
                )

                best_state, t_hist, v_hist = train_full(
                    model, train_ld, valid_ld, optimizer, criterion,
                    scheduler, max_epochs, config["patience"], target_idx,
                    label=f"{mode[:5]}-{strategy[:2]}"
                )

                model.load_state_dict(best_state)
                mse, mae, _, preds, trues = evaluate(
                    model, test_ld, criterion, target_idx
                )
                print(f"    TEST →  MSE: {mse:.6f}   MAE: {mae:.6f}")

                results.append({
                    "Dataset": dataset_name, "pred_len": pred_len,
                    "Modo": mode, "Estrategia": strategy,
                    "MSE (OT)": round(mse, 6), "MAE (OT)": round(mae, 6)
                })

                key = f"{dataset_name}_{mode}_{strategy}_pl{pred_len}"
                histories[key] = {"train": t_hist, "valid": v_hist}
                test_preds[key] = {
                    "preds": preds, "trues": trues, "target_idx": target_idx
                }

                del model, optimizer, scheduler
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    return results, histories, test_preds

# 7. Visualización de Resultados y Reportes Tabulares

Esta sección define las funciones encargadas de generar los gráficos analíticos y estructurar los reportes de datos para interpretar el comportamiento y la precisión de los modelos experimentales.

* **Curvas de Pérdida (`plot_loss_curves`):**  Construye una cuadrícula visual de la evolución de la pérdida (Loss) en entrenamiento y validación, crucial para diagnosticar el sobreajuste y verificar la parada temprana (*early stopping*).
* **Comparativa de Predicciones (`plot_predictions`):**  Genera gráficos de línea por horizonte temporal. Contrasta la señal real (*Ground Truth*) contra las proyecciones de los modelos inicializados desde cero (*Scratch*) y los adaptados (*Fine-Tuning*).
* **Tabla de Rendimiento (`format_comparison_table`):** Reestructura (*pivot*) el DataFrame de resultados para comparar de forma directa las métricas de error (MSE y MAE). Calcula automáticamente la mejora relativa ($\Delta\%$), donde un valor negativo indica matemáticamente la superioridad del enfoque de *Fine-Tuning*.

In [ ]:
# ============================================================================
# 7.1 GRÁFICAS
# ============================================================================

def plot_loss_curves(histories: dict, title_prefix: str = "",
                     save_path: str = "loss_curves.png"):
    """Grid de curvas Train vs Valid Loss."""
    n = len(histories)
    if n == 0:
        return
    cols = min(4, n)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    if n == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    for idx, (key, hist) in enumerate(histories.items()):
        if idx >= len(axes):
            break
        ax = axes[idx]
        epochs = range(1, len(hist["train"]) + 1)
        ax.plot(epochs, hist["train"], label="Train", linewidth=1.5,
                color="#1976D2")
        ax.plot(epochs, hist["valid"], label="Valid", linewidth=1.5,
                color="#E64A19")

        # Título legible desde la clave
        ax.set_title(key.replace("_", " | ").replace("pl", "PL="),
                     fontsize=8, fontweight='bold')
        ax.set_xlabel("Época", fontsize=7)
        ax.set_ylabel("Loss", fontsize=7)
        ax.legend(fontsize=6)
        ax.grid(True, alpha=0.3)

    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"{title_prefix} — Curvas de Pérdida", fontsize=13,
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[PLOT] {save_path}")


def plot_predictions(test_preds: dict, pred_lens: list,
                     dataset_name: str):
    """
    Un gráfico por horizonte, con Multivariado arriba y Univariado abajo.
    Cada panel muestra Ground Truth vs Scratch vs Fine-Tuning.
    """
    for pl in pred_lens:
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        fig.suptitle(f"{dataset_name} — Horizonte: {pl} pasos",
                     fontsize=14, fontweight='bold')

        for idx, mode in enumerate(["Multivariado", "Univariado"]):
            ax = axes[idx]
            key_s  = f"{dataset_name}_{mode}_Scratch_pl{pl}"
            key_ft = f"{dataset_name}_{mode}_Fine-Tuning_pl{pl}"

            if key_s not in test_preds:
                continue

            ti = test_preds[key_s]["target_idx"]

            # Primer paso de cada ventana → trayectoria continua
            gt     = test_preds[key_s]["trues"][:, 0, ti].numpy()
            pred_s = test_preds[key_s]["preds"][:, 0, ti].numpy()

            ax.plot(gt, label="Ground Truth", color="black",
                    linewidth=1.5, alpha=0.8)
            ax.plot(pred_s, label="Scratch", color="#2196F3",
                    linewidth=1.2, linestyle="--")

            if key_ft in test_preds:
                pred_ft = test_preds[key_ft]["preds"][:, 0, ti].numpy()
                ax.plot(pred_ft, label="Fine-Tuning (ETTh1→)",
                        color="#FF5722", linewidth=1.2, linestyle=":")

            ax.set_title(f"{mode}", fontsize=11)
            ax.set_xlabel("Ventana temporal")
            ax.set_ylabel("OT (Oil Temperature)")
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        save_path = f"predictions_{dataset_name}_pl{pl}.png"
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"[PLOT] {save_path}")

In [ ]:
# ============================================================================
# 7.2 TABLA COMPARATIVA
# ============================================================================

def format_comparison_table(results_df: pd.DataFrame):
    """
    Pivotea para comparar Scratch vs Fine-Tuning lado a lado,
    agrupado por Dataset, pred_len y Modo.
    """
    df = results_df.copy()
    pivot = df.pivot_table(
        index=["Dataset", "pred_len", "Modo"],
        columns="Estrategia",
        values=["MSE (OT)", "MAE (OT)"]
    )
    pivot.columns = [f"{m} — {s}" for m, s in pivot.columns]
    pivot = pivot.reset_index()

    # Columna de mejora relativa (%)  → negativo = FT es mejor
    if "MSE (OT) — Scratch" in pivot.columns and \
       "MSE (OT) — Fine-Tuning" in pivot.columns:
        pivot["MSE Δ% (FT vs Scratch)"] = (
            (pivot["MSE (OT) — Fine-Tuning"] - pivot["MSE (OT) — Scratch"])
            / pivot["MSE (OT) — Scratch"] * 100
        ).round(2)
    return pivot

In [ ]:
# ============================================================================
# 7.3 CARGA DE DATOS
# ============================================================================

def load_dataset(name: str) -> pd.DataFrame:
    """Descarga el dataset desde GitHub. Si falla, detiene la ejecución arrojando un error."""
    try:
        df = pd.read_csv(DATA_URLS[name])
        print(f"  [OK]   {name} descargado ({df.shape[0]:,} filas)")
        return df
    except Exception as e:
        print(f"  [ERROR] Falló la descarga de {name}. Revisa la URL o tu conexión.")
        raise e

# 8. Programa Principal (Orquestación del Experimento)



Esta es la función `main()` que actúa como el cerebro del proyecto, uniendo todas las piezas previamente definidas. Automatiza el flujo de trabajo de principio a fin de la siguiente manera:

* **Hiperparámetros:** Centraliza la configuración estructural y de entrenamiento. Define el contexto temporal (`seq_len=96`), las dimensiones del *Patching* y del Transformer, además de fijar las tasas de aprendizaje (usando un `lr` estándar para entrenar desde cero y un `lr_ft` más conservador para el *Fine-Tuning*).
* **Fase 1 (Pre-entrenamiento):** Ejecuta el aprendizaje base en el dataset origen (`ETTh1`) a través de múltiples horizontes de predicción (`pred_lens = [24, 96, 336, 720]`) para generar los pesos fundacionales.
* **Fase 2 (Transferencia):** Itera sobre los datasets destino aplicando la evaluación comparativa, midiendo si el modelo se beneficia de los conocimientos previos frente a un inicio aleatorio (*Scratch*).
* **Generación de Reportes:** Compila todas las métricas en un DataFrame de Pandas, muestra por consola el resumen del desempeño y exporta los datos crudos a archivos `.csv` (`comparison_table.csv` y `results_all.csv`) para análisis futuros.

In [ ]:
# ============================================================================
# 8. UNIÓN PRINCIPAL
# ============================================================================

def main():
    # ================================================================
    # HIPERPARÁMETROS
    # ================================================================
    config = {
        # Ventanas
        "seq_len": 96,
        # Patching
        "patch_len": 16,
        "stride": 8,
        # Transformer
        "d_model": 128,
        "n_heads": 4,
        "e_layers": 3,
        "d_ff": 256,
        "dropout": 0.2,
        # Scratch
        "batch_size": 32,
        "lr": 1e-3,
        "epochs": 30,
        "patience": 7,
        # Fine-Tuning Total (lr reducido, misma arquitectura)
        "lr_ft": 1e-4,
        "epochs_ft": 20,
    }

    pred_lens = [24, 96, 336, 720]

    # ================================================================
    # CARGAR TODOS LOS DATASETS
    # ================================================================
    print("\n[INFO] Cargando datasets ...")
    datasets = {}
    for name in [SOURCE_DATASET] + TARGET_DATASETS:
        datasets[name] = load_dataset(name)

    # ================================================================
    # FASE 1 — PRE-ENTRENAMIENTO EN ETTh1 (DATASET ORIGEN)
    # ================================================================
    print("\n" + "█" * 80)
    print("  FASE 1:  PRE-ENTRENAMIENTO EN ETTh1 (DATASET ORIGEN)")
    print("█" * 80)

    pretrained_weights, pretrain_hist, pretrain_metrics = pretrain_on_source(
        datasets[SOURCE_DATASET], pred_lens, config
    )

    # Gráfica de pérdida del pre-entrenamiento
    plot_loss_curves(pretrain_hist, title_prefix="ETTh1 (Pretrain)",
                     save_path="loss_pretrain_ETTh1.png")

    # ================================================================
    # FASE 2 — SCRATCH vs FINE-TUNING EN DATASETS DESTINO
    # ================================================================
    all_results = list(pretrain_metrics)  # incluir métricas de pretrain
    all_histories = dict(pretrain_hist)
    all_test_preds = {}

    for target_name in TARGET_DATASETS:
        print("\n" + "█" * 80)
        print(f"  FASE 2:  {target_name}  (Scratch vs Fine-Tuning desde ETTh1)")
        print("█" * 80)

        res, hist, tpreds = evaluate_on_target(
            datasets[target_name], target_name, pred_lens, config,
            pretrained_weights
        )
        all_results.extend(res)
        all_histories.update(hist)
        all_test_preds.update(tpreds)

        # Gráficas por dataset destino
        plot_loss_curves(hist, title_prefix=target_name,
                         save_path=f"loss_{target_name}.png")
        plot_predictions(tpreds, pred_lens, target_name)

    # ================================================================
    # TABLA COMPARATIVA GLOBAL
    # ================================================================
    results_df = pd.DataFrame(all_results)

    # Tabla solo de datasets destino (donde existe Scratch vs FT)
    target_results = results_df[
        results_df["Dataset"].isin(TARGET_DATASETS)
    ].copy()

    print("\n" + "=" * 110)
    print("  TABLA COMPARATIVA — SCRATCH vs FINE-TUNING (Transfer desde ETTh1)")
    print("=" * 110)
    if len(target_results) > 0:
        comparison = format_comparison_table(target_results)
        print(comparison.to_string(index=False))
        comparison.to_csv("comparison_table.csv", index=False)
        print("\n[INFO] Tabla guardada en: comparison_table.csv")
    else:
        comparison = pd.DataFrame()

    # Tabla completa (incluyendo pretrain)
    print("\n" + "=" * 110)
    print("  MÉTRICAS DE PRE-ENTRENAMIENTO (ETTh1)")
    print("=" * 110)
    pretrain_df = results_df[results_df["Dataset"] == SOURCE_DATASET]
    print(pretrain_df.to_string(index=False))

    # Guardar todo
    results_df.to_csv("results_all.csv", index=False)
    print("\n[INFO] Todos los resultados guardados en: results_all.csv")

    print("\n" + "█" * 80)
    print("  EJECUCIÓN COMPLETA")
    print("█" * 80)
    return results_df, comparison


# ============================================================================
if __name__ == "__main__":
    results_df, comparison = main()



[INFO] Cargando datasets ...
  [OK]   ETTh1 descargado (17,420 filas)
  [OK]   ETTh2 descargado (17,420 filas)
  [OK]   ETTm1 descargado (69,680 filas)
  [OK]   ETTm2 descargado (69,680 filas)

████████████████████████████████████████████████████████████████████████████████
  FASE 1:  PRE-ENTRENAMIENTO EN ETTh1 (DATASET ORIGEN)
████████████████████████████████████████████████████████████████████████████████

  ────────────────────────────────────────────────────────────
  Pre-entrenando para pred_len = 24
  ────────────────────────────────────────────────────────────

    Modo: Multivariado (7 canales)
    Parámetros: 436,376
      [Pretrain-Multi] Epoch   1/30  Train: 5.421689  Valid: 16.692336
      [Pretrain-Multi] Epoch   5/30  Train: 4.569344  Valid: 13.791557
      [Pretrain-Multi] Epoch  10/30  Train: 4.230277  Valid: 17.066477
      [Pretrain-Multi] Early stopping en epoch 10
    TEST ETTh1 →  MSE: 4.514668   MAE: 1.620905

    Modo: Univariado (1 canal)
    Parámetros: 436,37